# Poseable object

Example of displaying a ghost molecule that can be reposed in 3d space.

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.jupyter.utilities import make_id_generator
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client cursors")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [5]:
from nanover.utilities.transforms import Transform

get_new_object_id = make_id_generator("object.")

OBJECT_TRANSFORMS = {}

def make_new_object():
    id = get_new_object_id()
    set_object_transform(id, Transform.identity())

    utilities.objects.update_shape(id, position=[0.0, 0.0, 0.0], color=[1.0, 0.0, 0.0, 1.0], size=1, parent=id)
    for z in range(2):
        for y in range(2):
            for x in range(2):
                utilities.objects.update_shape(f"{id}.{x}{y}{z}", position=[x-.5, y-.5, z-.5], color=[1.0-x*.5, 1.0-y*.5, 1.0-z*.5, 1.0], size=.5, parent=id)

    return id

def set_object_transform(id, transform):
    OBJECT_TRANSFORMS[id] = transform
    utilities.transforms.update_transform(id, transform=OBJECT_TRANSFORMS[id])


make_new_object()

'object.0'

In [11]:
from MDAnalysis.lib.transformations import translation_matrix

set_object_transform("object.0", Transform.from_parent_to_local_matrix(translation_matrix([0, 0, 5])))

In [4]:
import numpy as np
from nanover.jupyter import Mode


OBJECT_RADIUS = 1
OBJECT_COLOR = [1.0, 0.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 1.0, 1.0, 0.5]
CURSOR_RADIUS = 0

# mapping of cursor to currently grabbed object if any
CURSOR_GRABBED_OBJECT_ID = {}
# mapping of object id to object position
OBJECT_POSITIONS = {}


def redraw_object(object_id):
    utilities.objects.update_shape(object_id, position=OBJECT_POSITIONS[object_id], color=OBJECT_COLOR, size=OBJECT_RADIUS)


def intersect_objects(position):
    for object_id, object_pos in OBJECT_POSITIONS.items():
        if np.linalg.norm(np.subtract(position, object_pos)) < OBJECT_RADIUS + CURSOR_RADIUS:
            return object_id
    return None


# put some objects in the scene
for i in range(4):
    OBJECT_POSITIONS[str(i)] = [i, i, i]
    redraw_object(str(i))


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_objects(next_pos)
        available = hovered not in CURSOR_GRABBED_OBJECT_ID.values()

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            CURSOR_GRABBED_OBJECT_ID[key] = hovered

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed something, update its position
        if key in CURSOR_GRABBED_OBJECT_ID:
            object_id = CURSOR_GRABBED_OBJECT_ID[key]
            OBJECT_POSITIONS[object_id] = next_pos
            redraw_object(object_id)

        # show/remove hover graphic is this cursor hovers something
        hovered = intersect_objects(next_pos)
        if hovered is None:
            utilities.objects.update_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=OBJECT_POSITIONS[hovered], color=HOVER_COLOR, size=OBJECT_RADIUS * 1.1)


utilities.use_interaction_modes()
utilities.add_interaction_mode(MoveObjectMode, "move object", icon="✊")